# 05. 3D 회전 표현

3D 회전을 표현하는 방법이 세 가지나 있는 이유: 각각 장단점이 다르기 때문.

| 표현 | 자유도 | 특이점 | ROS2 |
|------|--------|--------|------|
| 회전행렬 $R \in SO(3)$ | 9개 원소 (실제론 3DoF) | 없음 | 내부 계산 |
| 오일러각 (RPY) | 3개 | **짐벌락** | 가독성용 |
| 쿼터니언 $q \in \mathbb{H}$ | 4개 (단위 크기 조건) | 없음 | **표준** |

**로보틱스 연결:**
- ROS2 `geometry_msgs/Quaternion` → 쿼터니언이 표준
- 순기구학에서 각 관절 좌표계 회전 = $R_z(\theta_i)$ 합성
- 짐벌락은 pitch가 ±90°일 때 발생 → 드론/로봇에서 회피해야 함

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import os
os.makedirs('assets', exist_ok=True)

plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False

def Rx(deg):
    a = np.radians(deg)
    return np.array([[1,0,0],[0,np.cos(a),-np.sin(a)],[0,np.sin(a),np.cos(a)]])

def Ry(deg):
    a = np.radians(deg)
    return np.array([[np.cos(a),0,np.sin(a)],[0,1,0],[-np.sin(a),0,np.cos(a)]])

def Rz(deg):
    a = np.radians(deg)
    return np.array([[np.cos(a),-np.sin(a),0],[np.sin(a),np.cos(a),0],[0,0,1]])

def draw_axes(ax, R, origin=np.zeros(3), scale=1.0, alpha=1.0):
    colors = ['#E85D24', '#1D9E75', '#534AB7']
    labels = ['X', 'Y', 'Z']
    for i, (color, label) in enumerate(zip(colors, labels)):
        vec = R[:, i] * scale
        ax.quiver(*origin, *vec, color=color, lw=2.5, arrow_length_ratio=0.2, alpha=alpha)
        ax.text(*(origin + vec * 1.2), label, color=color, fontsize=10, fontweight='bold')

## 1. 회전행렬 $SO(3)$ — 성질 확인

$R \in SO(3)$ 의 두 가지 핵심 성질:
1. $R^T R = I$ (직교 행렬 → 역행렬 = 전치)
2. $\det(R) = 1$ (방향 보존, -1이면 반전)

좌표 변환에서 $R^{-1} = R^T$ 가 성립하기 때문에 역변환 계산이 싸다.

In [ ]:
# 세 기본 회전 시각화
angles = [30, 45, 60]
fig = plt.figure(figsize=(15, 5))

for idx, (func, name, angle) in enumerate([(Rx, 'Rx', angles[0]),
                                            (Ry, 'Ry', angles[1]),
                                            (Rz, 'Rz', angles[2])]):
    ax = fig.add_subplot(1, 3, idx+1, projection='3d')
    R = func(angle)

    # 원래 축 (흐리게)
    draw_axes(ax, np.eye(3), scale=0.8, alpha=0.2)
    # 회전된 축
    draw_axes(ax, R, scale=0.8)

    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-1.2, 1.2); ax.set_zlim(-1.2, 1.2)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    det = np.linalg.det(R)
    ortho = np.linalg.norm(R.T @ R - np.eye(3))
    ax.set_title(f'{name}({angle}°)\ndet={det:.3f}, ||RᵀR-I||={ortho:.1e}', fontsize=10)

plt.suptitle('기본 3D 회전행렬 — det=1, 직교 성질 확인', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('assets/05_rotation_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# ZYX 오일러각 (ROS RPY 순서)
roll, pitch, yaw = 20, 35, 50
R_rpy = Rz(yaw) @ Ry(pitch) @ Rx(roll)
print(f'RPY({roll}°, {pitch}°, {yaw}°) 회전행렬:')
print(R_rpy.round(4))
print(f'det = {np.linalg.det(R_rpy):.6f}')
print(f'||RᵀR - I|| = {np.linalg.norm(R_rpy.T @ R_rpy - np.eye(3)):.2e}')

## 2. 짐벌락 (Gimbal Lock)

오일러각에서 pitch가 ±90° 가 되면 roll과 yaw가 같은 축을 돌게 된다.
→ 자유도 1개 상실 → 특정 방향으로 회전 불가능.

$R_{ZYX}(\alpha, \pm90°, \gamma) = R_Z(\alpha \pm \gamma) \cdot R_X(0)$

드론이나 로봇팔이 수직으로 들어갈 때 이 문제가 생긴다.

In [ ]:
fig = plt.figure(figsize=(15, 5))

pitch_cases = [0, 45, 89]
test_vec = np.array([1., 0., 0.])

for idx, pitch in enumerate(pitch_cases):
    ax = fig.add_subplot(1, 3, idx+1, projection='3d')

    # roll과 yaw를 여러 값으로 변화시킬 때 끝점 분포
    rolls  = np.linspace(0, 360, 36)
    yaws   = np.linspace(0, 360, 36)
    pts = []
    for r in rolls:
        for y in yaws:
            R = Rz(y) @ Ry(pitch) @ Rx(r)
            pts.append(R @ test_vec)
    pts = np.array(pts)

    ax.scatter(pts[:,0], pts[:,1], pts[:,2], s=3, alpha=0.3, color='#534AB7')
    draw_axes(ax, np.eye(3), scale=0.9, alpha=0.15)

    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')

    unique = len(set([tuple(p.round(2)) for p in pts]))
    label = '짐벌락!' if pitch >= 88 else ''
    ax.set_title(f'pitch={pitch}°  {label}\n도달 가능 방향 수: ~{unique}', fontsize=10,
                 color='#E85D24' if pitch >= 88 else 'black')

plt.suptitle('짐벌락 — pitch→90° 되면 도달 가능 방향이 줄어든다', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('assets/05_gimbal_lock.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 쿼터니언 — 짐벌락 없는 회전 표현

$$q = w + xi + yj + zk, \quad |q| = 1$$

축-각도 표현 $(\hat{n}, \theta)$ 으로부터:

$$q = \left[\cos\frac{\theta}{2},\ \hat{n}\sin\frac{\theta}{2}\right]$$

ROS2에서 `geometry_msgs/Quaternion` 을 그냥 쓰면 되는데,
내부적으로 어떻게 돌아가는지 알아야 디버깅이 된다.

In [ ]:
def axis_angle_to_quat(axis, deg):
    # 축-각도 to 쿼터니언 [w, x, y, z]
    n = np.array(axis, dtype=float)
    n /= np.linalg.norm(n)
    a = np.radians(deg) / 2
    return np.array([np.cos(a), *(n * np.sin(a))])

def quat_to_R(q):
    w, x, y, z = q
    return np.array([
        [1-2*(y**2+z**2),  2*(x*y-w*z),    2*(x*z+w*y)],
        [2*(x*y+w*z),    1-2*(x**2+z**2),  2*(y*z-w*x)],
        [2*(x*z-w*y),    2*(y*z+w*x),    1-2*(x**2+y**2)]
    ])

def quat_mul(q1, q2):
    # 쿼터니언 곱 (회전 합성)
    w1,x1,y1,z1 = q1; w2,x2,y2,z2 = q2
    return np.array([
        w1*w2 - x1*x2 - y1*y2 - z1*z2,
        w1*x2 + x1*w2 + y1*z2 - z1*y2,
        w1*y2 - x1*z2 + y1*w2 + z1*x2,
        w1*z2 + x1*y2 - y1*x2 + z1*w2
    ])

# 예: Z축 90° 회전
q_z90 = axis_angle_to_quat([0,0,1], 90)
R_q   = quat_to_R(q_z90)
R_mat = Rz(90)

print(f'q_z90 = {q_z90.round(4)}')
print(f'쿼터니언→R:\n{R_q.round(4)}')
print(f'직접 Rz(90°):\n{R_mat.round(4)}')
print(f'차이: {np.linalg.norm(R_q - R_mat):.2e}')

# 구면 선형 보간 (SLERP) — 쿼터니언의 장점
def slerp(q1, q2, t):
    dot = np.clip(np.dot(q1, q2), -1, 1)
    if dot < 0:
        q2 = -q2; dot = -dot
    if dot > 0.9995:
        return q1 + t*(q2-q1)
    omega = np.arccos(dot)
    return (np.sin((1-t)*omega)*q1 + np.sin(t*omega)*q2) / np.sin(omega)

q_start = axis_angle_to_quat([0,0,1], 0)
q_end   = axis_angle_to_quat([1,1,0], 90)

fig = plt.figure(figsize=(12, 5))
ax = fig.add_subplot(121, projection='3d')
ts = np.linspace(0, 1, 10)
colors_slerp = plt.cm.viridis(ts)
for t_val, c in zip(ts, colors_slerp):
    q = slerp(q_start, q_end, t_val)
    q /= np.linalg.norm(q)
    R = quat_to_R(q)
    draw_axes(ax, R, scale=0.6, alpha=0.6)

ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
ax.set_title('SLERP — 쿼터니언 구면 보간\n(자연스러운 회전 경로)', fontsize=10)

# 세 표현 비교 바 차트
ax2 = fig.add_subplot(122)
methods = ['회전행렬\nSO(3)', '오일러각\nRPY', '쿼터니언\n[w,x,y,z]']
params  = [9, 3, 4]
gimbal  = [0, 1, 0]   # 짐벌락 있으면 1
colors_bar = ['#1D9E75', '#E85D24', '#534AB7']
bars = ax2.bar(methods, params, color=colors_bar, width=0.4)
for bar, g, m in zip(bars, gimbal, methods):
    if g:
        ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                 '⚠ 짐벌락', ha='center', fontsize=9, color='#E85D24')
ax2.set_ylabel('파라미터 수'); ax2.set_ylim(0, 12)
ax2.set_title('3D 회전 표현 비교', fontsize=11)
ax2.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('assets/05_quaternion.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 표현 | 파라미터 | 짐벌락 | 보간 | 용도 |
|------|----------|--------|------|------|
| 회전행렬 $R$ | 9 (실제 3DoF) | 없음 | 복잡 | 내부 계산 |
| 오일러각 | 3 | **있음** | 쉬움 | 가독성 |
| 쿼터니언 | 4 (크기 1 조건) | 없음 | SLERP | **ROS2 표준** |

**다음 노트북:** `06_least_squares.ipynb` — 최소제곱법